# Kaggle Playground Series S6E5 — Predict F1 Pit Stops
### Author: Ruide Yin

## Stage 3-5: Full Pipeline — Tuning & Ensemble

Continues from Stage 2 - feature engineering and model training. 

### 3.0 Setup, Load Data & Load Models

In [5]:
# As NYU HPC always unreasonably kills my session of this hyperparameter tuning notebook, we will use Kaggle resource.
import shutil
shutil.copytree('/kaggle/input/datasets/ruideyindzdz/pit-stop-outputs', '/kaggle/working/outputs')

'/kaggle/working/outputs'

In [6]:
import pandas as pd, numpy as np, os, time, warnings, gc, joblib, pickle
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
warnings.filterwarnings('ignore')

SEED = 12138
N_FOLDS = 5
TARGET = 'PitNextLap'

X = np.load('./outputs/X.npy')
y = np.load('./outputs/y.npy')
X_test = np.load('./outputs/X_test.npy')
X_scaled = np.load('./outputs/X_scaled.npy')
X_test_scaled = np.load('./outputs/X_test_scaled.npy')
X_filled = np.load('./outputs/X_filled.npy')
X_test_filled = np.load('./outputs/X_test_filled.npy')
test_ids = np.load('./outputs/test_ids.npy')
FEATURES = pickle.load(open('./outputs/features_list.pkl', 'rb'))
results = pickle.load(open('./outputs/baseline_results.pkl', 'rb'))
scale_pos = pickle.load(open('./outputs/scale_pos.pkl', 'rb'))
scaler = joblib.load('./outputs/scaler.pkl')
print(f"Loaded: X {X.shape}, y {y.shape}, {len(results)} baseline models")
print(f"scale_pos_weight = {scale_pos:.2f}")

Loaded: X (439140, 42), y (439140,), 6 baseline models
scale_pos_weight = 4.03


In [7]:
# ── Training functions (copied from 02_train.ipynb) ──

def train_lgb_model(params, X_tr, y_tr, X_te, name, n_folds=N_FOLDS):
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        dtrain = lgb.Dataset(X_tr[trn_idx], y_tr[trn_idx])
        dval   = lgb.Dataset(X_tr[val_idx], y_tr[val_idx], reference=dtrain)
        model = lgb.train(params, dtrain, num_boost_round=3000,
                          valid_sets=[dval], callbacks=[
                              lgb.early_stopping(100, verbose=False),
                              lgb.log_evaluation(0)])
        oof[val_idx] = model.predict(X_tr[val_idx])
        test_preds += model.predict(X_te) / n_folds
    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed


def train_xgb_model(params, X_tr, y_tr, X_te, name, n_folds=N_FOLDS):
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        dtrain = xgb.DMatrix(X_tr[trn_idx], y_tr[trn_idx], feature_names=FEATURES)
        dval   = xgb.DMatrix(X_tr[val_idx], y_tr[val_idx], feature_names=FEATURES)
        model = xgb.train(params, dtrain, num_boost_round=3000,
                          evals=[(dval, 'val')],
                          early_stopping_rounds=100, verbose_eval=0)
        oof[val_idx] = model.predict(dval)
        dtest = xgb.DMatrix(X_te, feature_names=FEATURES)
        test_preds += model.predict(dtest) / n_folds
    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed


class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128), dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


def train_mlp_model(X_tr, y_tr, X_te, name, hidden_dims=(256, 128),
                    lr=1e-3, dropout=0.3, weight_decay=1e-4, batch_size=4096,
                    epochs=50, n_folds=N_FOLDS):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    pos_w = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    pos_weight = torch.tensor([pos_w], dtype=torch.float32).to(device)
    X_te_t = torch.tensor(X_te, dtype=torch.float32).to(device)
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        X_t = torch.tensor(X_tr[trn_idx], dtype=torch.float32)
        y_t = torch.tensor(y_tr[trn_idx], dtype=torch.float32)
        X_v = torch.tensor(X_tr[val_idx], dtype=torch.float32).to(device)
        train_ds = TensorDataset(X_t, y_t)
        train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
        model = MLPClassifier(X_tr.shape[1], hidden_dims, dropout).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        best_auc, patience, best_state = 0, 0, None
        for epoch in range(epochs):
            model.train()
            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)
                loss = criterion(model(xb).squeeze(), yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            scheduler.step()
            model.eval()
            with torch.no_grad():
                val_logits = model(X_v).squeeze().cpu().numpy()
                val_preds = 1 / (1 + np.exp(-val_logits))
                auc = roc_auc_score(y_tr[val_idx], val_preds)
            if auc > best_auc:
                best_auc = auc; patience = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience += 1
                if patience >= 7: break
        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            oof[val_idx] = 1 / (1 + np.exp(-model(X_v).squeeze().cpu().numpy()))
            test_preds += (1 / (1 + np.exp(-model(X_te_t).squeeze().cpu().numpy()))) / n_folds
    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed, model

print("Training functions loaded.")

Training functions loaded.


## Part 3: Hyperparameter Tuning (Optuna)

Inner CV uses StratifiedKFold 3-fold for speed. After tuning, final models retrained with 5-fold.

### 3.1 LightGBM Tuning (60 trials)

In [8]:
def lgb_objective(trial):
    params = {
        'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
        'n_jobs': -1, 'random_state': SEED,
        'scale_pos_weight': scale_pos,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'max_depth':         trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq':    1,
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []
    for trn_idx, val_idx in skf.split(X, y):
        dtrain = lgb.Dataset(X[trn_idx], y[trn_idx])
        dval   = lgb.Dataset(X[val_idx], y[val_idx], reference=dtrain)
        model = lgb.train(params, dtrain, num_boost_round=2000,
                          valid_sets=[dval], callbacks=[
                              lgb.early_stopping(50, verbose=False),
                              lgb.log_evaluation(0)
                          ])
        preds = model.predict(X[val_idx])
        aucs.append(roc_auc_score(y[val_idx], preds))
    return np.mean(aucs)

t0 = time.time()
study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgb.optimize(lgb_objective, n_trials=60, show_progress_bar=True)
print(f"\nLGB best trial AUC: {study_lgb.best_value:.6f}  ({time.time()-t0:.0f}s)")
print(f"Best params: {study_lgb.best_params}")

  0%|          | 0/60 [00:00<?, ?it/s]


LGB best trial AUC: 0.949078  (18993s)
Best params: {'learning_rate': 0.013868304514548416, 'num_leaves': 248, 'max_depth': 11, 'min_child_samples': 143, 'reg_alpha': 0.0037203423384150596, 'reg_lambda': 7.411872939737794, 'colsample_bytree': 0.5800124909196928, 'subsample': 0.8637226999398172}


### 3.2 XGBoost Tuning (150 trials)

In [9]:
def xgb_objective(trial):
    params = {
        'objective': 'binary:logistic', 'eval_metric': 'auc',
        'tree_method': 'hist', 'device': 'cuda', 'seed': SEED,
        'scale_pos_weight': scale_pos,
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 200),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []
    for trn_idx, val_idx in skf.split(X, y):
        dtrain = xgb.DMatrix(X[trn_idx], y[trn_idx], feature_names=FEATURES)
        dval   = xgb.DMatrix(X[val_idx], y[val_idx], feature_names=FEATURES)
        model = xgb.train(params, dtrain, num_boost_round=2000,
                          evals=[(dval, 'val')],
                          early_stopping_rounds=50, verbose_eval=0)
        preds = model.predict(dval)
        aucs.append(roc_auc_score(y[val_idx], preds))
    return np.mean(aucs)

t0 = time.time()
study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(xgb_objective, n_trials=150, show_progress_bar=True)
print(f"\nXGB best trial AUC: {study_xgb.best_value:.6f}  ({time.time()-t0:.0f}s)")
print(f"Best params: {study_xgb.best_params}")

  0%|          | 0/150 [00:00<?, ?it/s]


XGB best trial AUC: 0.949381  (7980s)
Best params: {'learning_rate': 0.012045848421624046, 'max_depth': 12, 'min_child_weight': 42, 'reg_alpha': 0.0011787069074640107, 'reg_lambda': 0.06398282986534404, 'colsample_bytree': 0.5223327865824058, 'subsample': 0.8692519092416161, 'gamma': 2.1741657717467615}


### 3.3 MLP Tuning (5 trials)

In [14]:
def mlp_objective(trial):
    n_layers = trial.suggest_int('n_layers', 1, 3)
    hidden_dims = tuple(
        trial.suggest_categorical(f'dim_{i}', [64, 128, 256, 512])
        for i in range(n_layers)
    )
    lr      = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    wd      = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    bs      = trial.suggest_categorical('batch_size', [2048, 4096, 8192])

    device = torch.device('cpu')
    pos_w = (y == 0).sum() / max((y == 1).sum(), 1)
    pos_weight = torch.tensor([pos_w], dtype=torch.float32).to(device)

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []
    X_s = X_scaled.astype(np.float32)

    for trn_idx, val_idx in skf.split(X_s, y):
        X_t = torch.tensor(X_s[trn_idx], dtype=torch.float32)
        y_t = torch.tensor(y[trn_idx], dtype=torch.float32)
        X_v = torch.tensor(X_s[val_idx], dtype=torch.float32).to(device)
        train_dl = DataLoader(TensorDataset(X_t, y_t), batch_size=bs, shuffle=True, drop_last=True)

        model = MLPClassifier(X_s.shape[1], hidden_dims, dropout).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        best_auc_fold = 0
        for epoch in range(30):
            model.train()
            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)
                loss = criterion(model(xb).squeeze(), yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            model.eval()
            with torch.no_grad():
                vp = 1 / (1 + np.exp(-model(X_v).squeeze().cpu().numpy()))
                a = roc_auc_score(y[val_idx], vp)
            best_auc_fold = max(best_auc_fold, a)
        aucs.append(best_auc_fold)
    return np.mean(aucs)

t0 = time.time()
study_mlp = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_mlp.optimize(mlp_objective, n_trials=5, show_progress_bar=True)
print(f"\nMLP best trial AUC: {study_mlp.best_value:.6f}  ({time.time()-t0:.0f}s)")
print(f"Best params: {study_mlp.best_params}")

  0%|          | 0/5 [00:00<?, ?it/s]


MLP best trial AUC: 0.940970  (6185s)
Best params: {'n_layers': 3, 'dim_0': 512, 'dim_1': 512, 'dim_2': 64, 'lr': 0.0009015709485637984, 'dropout': 0.3511248427752316, 'weight_decay': 7.443398568486106e-06, 'batch_size': 4096}


### 3.4 Retrain Tuned Models (5-Fold) + Keep Untuned RF & CatBoost

In [16]:
# ── Kaggle P100 GPU incompatible with current PyTorch build, force CPU for MLP ──

class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128), dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


def train_mlp_model(X_tr, y_tr, X_te, name, hidden_dims=(256, 128),
                    lr=1e-3, dropout=0.3, weight_decay=1e-4, batch_size=4096,
                    epochs=30, n_folds=N_FOLDS): # adjust epochs not to reach the time limit
    device = torch.device('cpu')  # forced CPU due to Kaggle P100 compatibility issue
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    pos_w = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    pos_weight = torch.tensor([pos_w], dtype=torch.float32).to(device)
    X_te_t = torch.tensor(X_te, dtype=torch.float32).to(device)
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        X_t = torch.tensor(X_tr[trn_idx], dtype=torch.float32)
        y_t = torch.tensor(y_tr[trn_idx], dtype=torch.float32)
        X_v = torch.tensor(X_tr[val_idx], dtype=torch.float32).to(device)
        train_ds = TensorDataset(X_t, y_t)
        train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
        model = MLPClassifier(X_tr.shape[1], hidden_dims, dropout).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        best_auc, patience, best_state = 0, 0, None
        for epoch in range(epochs):
            model.train()
            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)
                loss = criterion(model(xb).squeeze(), yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            scheduler.step()
            model.eval()
            with torch.no_grad():
                val_logits = model(X_v).squeeze().cpu().numpy()
                val_preds = 1 / (1 + np.exp(-val_logits))
                auc = roc_auc_score(y_tr[val_idx], val_preds)
            if auc > best_auc:
                best_auc = auc; patience = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience += 1
                if patience >= 7: break
        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            oof[val_idx] = 1 / (1 + np.exp(-model(X_v).squeeze().cpu().numpy()))
            test_preds += (1 / (1 + np.exp(-model(X_te_t).squeeze().cpu().numpy()))) / n_folds
    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed, model

print("MLPClassifier & train_mlp_model redefined with CPU device.")

MLPClassifier & train_mlp_model redefined with CPU device.


In [17]:
# ── Tuned LightGBM ──
lgb_tuned_params = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'n_jobs': -1, 'random_state': SEED,
    'scale_pos_weight': scale_pos,
    'subsample_freq': 1,
    **study_lgb.best_params
}
oof_lgb_t, tp_lgb_t, auc_lgb_t, t_lgb_t = train_lgb_model(lgb_tuned_params, X, y, X_test, 'LGB_tuned')
results['LGB_tuned'] = (oof_lgb_t, tp_lgb_t, auc_lgb_t, t_lgb_t)

# ── Tuned XGBoost ──
xgb_tuned_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'tree_method': 'hist', 'device': 'cuda', 'seed': SEED,
    'scale_pos_weight': scale_pos,
    **study_xgb.best_params
}
oof_xgb_t, tp_xgb_t, auc_xgb_t, t_xgb_t = train_xgb_model(xgb_tuned_params, X, y, X_test, 'XGB_tuned')
results['XGB_tuned'] = (oof_xgb_t, tp_xgb_t, auc_xgb_t, t_xgb_t)

# ── Tuned MLP ──
bp = study_mlp.best_params
n_layers = bp['n_layers']
mlp_hidden = tuple(bp[f'dim_{i}'] for i in range(n_layers))
oof_mlp_t, tp_mlp_t, auc_mlp_t, t_mlp_t, mlp_tuned_model = train_mlp_model(
    X_scaled.astype(np.float32), y, X_test_scaled.astype(np.float32), 'MLP_tuned',
    hidden_dims=mlp_hidden, lr=bp['lr'], dropout=bp['dropout'],
    weight_decay=bp['weight_decay'], batch_size=bp['batch_size'], epochs=30
)
results['MLP_tuned'] = (oof_mlp_t, tp_mlp_t, auc_mlp_t, t_mlp_t)

# ── Untuned RF & CatBoost are kept from baseline ──
# Already in results as RF_baseline, CAT_baseline

[LGB_tuned] OOF AUC = 0.949736  |  Time = 1066.8s
[XGB_tuned] OOF AUC = 0.950023  |  Time = 152.5s
[MLP_tuned] OOF AUC = 0.940042  |  Time = 2246.0s


In [18]:
# ── Print full comparison table ──
print("\n" + "="*70)
print("ALL MODELS — BASELINE vs TUNED")
print("="*70)
comp_df = pd.DataFrame({
    'Model': list(results.keys()),
    'OOF_AUC': [v[2] for v in results.values()],
    'Time_s': [v[3] for v in results.values()],
}).sort_values('OOF_AUC', ascending=False).reset_index(drop=True)
comp_df.index = comp_df.index + 1
comp_df.index.name = 'Rank'
print(comp_df.to_string())


ALL MODELS — BASELINE vs TUNED
             Model   OOF_AUC       Time_s
Rank                                     
1        XGB_tuned  0.950023   152.515054
2        LGB_tuned  0.949736  1066.766284
3     XGB_baseline  0.948580    30.805804
4     LGB_baseline  0.948450   129.654911
5     CAT_baseline  0.947685    92.901379
6     MLP_baseline  0.941555   777.380629
7        MLP_tuned  0.940042  2245.957760
8      RF_baseline  0.938378   377.944832
9      LR_baseline  0.881202    49.939114


## Part 4: Ensemble

4.1 Filter models by AUC threshold (98% of max AUC).   
4.2 Weighted average (softmax of AUC).   
4.3 Stacking with Logistic Regression meta-learner.   
4.4 Pick the best.

### 4.1 Model Selection & Ensemble

In [25]:
# ── 4.1 Select top-3 models (XGB + LGB + CAT) ──
selected = ['XGB_tuned', 'LGB_tuned', 'CAT_baseline']
print(f"Selected models ({len(selected)}):")
for n in selected:
    print(f"  {n}: AUC = {results[n][2]:.6f}")

Selected models (3):
  XGB_tuned: AUC = 0.950023
  LGB_tuned: AUC = 0.949736
  CAT_baseline: AUC = 0.947685


In [26]:
# ── 4.2 Weighted Average (softmax of AUC) ──
aucs_sel = np.array([results[n][2] for n in selected])
# Softmax with temperature to sharpen
weights = np.exp(aucs_sel * 100) / np.exp(aucs_sel * 100).sum()
print("Ensemble weights:")
for n, w in zip(selected, weights):
    print(f"  {n}: {w:.4f}")

oof_wa = sum(results[n][0] * w for n, w in zip(selected, weights))
test_wa = sum(results[n][1] * w for n, w in zip(selected, weights))
auc_wa = roc_auc_score(y, oof_wa)
print(f"\nWeighted Average OOF AUC: {auc_wa:.6f}")

Ensemble weights:
  XGB_tuned: 0.3619
  LGB_tuned: 0.3517
  CAT_baseline: 0.2865

Weighted Average OOF AUC: 0.949902


In [27]:
# ── 4.3 Stacking with Logistic Regression meta-learner (5-Fold) ──
oof_stack_X = np.column_stack([results[n][0] for n in selected])
test_stack_X = np.column_stack([results[n][1] for n in selected])

oof_stack_pred = np.zeros(len(y))
test_stack_pred = np.zeros(test_stack_X.shape[0])
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for trn_idx, val_idx in skf.split(oof_stack_X, y):
    meta = LogisticRegression(max_iter=1000, C=1.0)
    meta.fit(oof_stack_X[trn_idx], y[trn_idx])
    oof_stack_pred[val_idx] = meta.predict_proba(oof_stack_X[val_idx])[:, 1]
    test_stack_pred += meta.predict_proba(test_stack_X)[:, 1] / N_FOLDS

auc_stack = roc_auc_score(y, oof_stack_pred)
print(f"Stacking OOF AUC: {auc_stack:.6f}")

Stacking OOF AUC: 0.950024


In [28]:
# ── 4.4 Pick the best ensemble method ──
print(f"\nWeighted Average AUC: {auc_wa:.6f}")
print(f"Stacking AUC:         {auc_stack:.6f}")

if auc_stack > auc_wa:
    final_test_preds = test_stack_pred
    final_method = 'Stacking'
    final_auc = auc_stack
else:
    final_test_preds = test_wa
    final_method = 'Weighted Average'
    final_auc = auc_wa

print(f"\n>>> Using {final_method} (AUC = {final_auc:.6f}) for submission")


Weighted Average AUC: 0.949902
Stacking AUC:         0.950024

>>> Using Stacking (AUC = 0.950024) for submission


## Part 5: Save Artifacts

In [31]:
# ── Save top-3 ensemble results ──
# OOF & test for selected models only
for name in selected:
    oof, tp, auc, t = results[name][:4]
    np.save(f'./outputs/oof_{name}.npy', oof)
    np.save(f'./outputs/test_{name}.npy', tp)

# Ensemble predictions
np.save('./outputs/oof_ensemble_wa.npy', oof_wa)
np.save('./outputs/test_ensemble_wa.npy', test_wa)
np.save('./outputs/oof_ensemble_stack.npy', oof_stack_pred)
np.save('./outputs/test_ensemble_stack.npy', test_stack_pred)
np.save('./outputs/test_final.npy', final_test_preds)

# Ensemble config
joblib.dump({
    'selected': selected,
    'weights': weights.tolist(),
    'method': final_method,
    'final_auc': final_auc,
}, './outputs/ensemble_config.pkl')

# Tuned params
joblib.dump(study_lgb.best_params, './outputs/lgb_best_params.pkl')
joblib.dump(study_xgb.best_params, './outputs/xgb_best_params.pkl')
joblib.dump(study_mlp.best_params, './outputs/mlp_best_params.pkl')

print("Cleaned and saved.")
print(os.listdir('./outputs/'))

Cleaned and saved.
['oof_XGB_tuned.npy', 'X_test_filled.npy', 'train_fe.csv', 'test_CAT_baseline.npy', 'X_test.npy', 'oof_CAT_baseline.npy', 'oof_ensemble_wa.npy', 'test_fe.csv', 'features_list.pkl', 'oof_LGB_tuned.npy', 'X_test_scaled.npy', 'X_scaled.npy', 'baseline_results.pkl', 'mlp_tuned.pt', 'test_XGB_tuned.npy', 'ensemble_config.pkl', 'X.npy', 'lgb_best_params.pkl', 'mlp_best_params.pkl', 'scaler.pkl', 'xgb_best_params.pkl', 'test_ensemble_stack.npy', 'y.npy', 'test_LGB_tuned.npy', 'X_filled.npy', 'scale_pos.pkl', 'test_final.npy', 'test_ensemble_wa.npy', 'oof_ensemble_stack.npy']


### Final Summary

In [32]:
# ── Final comparison table ──
print("="*75)
print("FINAL MODEL RANKING")
print("="*75)
all_results = dict(results)
all_results['Ensemble_WA'] = (oof_wa, test_wa, auc_wa, 0)
all_results['Ensemble_Stack'] = (oof_stack_pred, test_stack_pred, auc_stack, 0)

final_df = pd.DataFrame({
    'Model': list(all_results.keys()),
    'OOF_AUC': [v[2] for v in all_results.values()],
}).sort_values('OOF_AUC', ascending=False).reset_index(drop=True)
final_df.index = final_df.index + 1
final_df.index.name = 'Rank'
print(final_df.to_string())
print(f"\n>>> Submitted with: {final_method} | AUC = {final_auc:.6f}")

FINAL MODEL RANKING
               Model   OOF_AUC
Rank                          
1     Ensemble_Stack  0.950024
2          XGB_tuned  0.950023
3        Ensemble_WA  0.949902
4          LGB_tuned  0.949736
5       XGB_baseline  0.948580
6       LGB_baseline  0.948450
7       CAT_baseline  0.947685
8       MLP_baseline  0.941555
9          MLP_tuned  0.940042
10       RF_baseline  0.938378
11       LR_baseline  0.881202

>>> Submitted with: Stacking | AUC = 0.950024
